# Quant Risk Core: Comprehensive Analysis Suite
This notebook provides a deep-dive into every module of the `quant_risk_core` repository, covering derivatives pricing, volatility modeling, portfolio risk, and credit risk.

### Environment Setup
This cell ensures all required libraries are installed in the current environment. It is designed for portability across local machines and cloud platforms like Google Colab.

In [1]:
# Install dependencies silently if not present
import sys
!{sys.executable} -m pip install -q numpy scipy pandas arch statsmodels numba plotly yfinance


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import scipy.stats as stats

## 1. Derivatives Pricing: Black-Scholes & Greeks
The Black-Scholes model calculates the theoretical price of European options by modeling the underlying asset as a geometric Brownian motion. It is used to derive Greeks, which measure how sensitive an option's price is to changes in spot price, time, and volatility.

In [3]:
from market_risk.pricing import BlackScholesEngine

S = np.linspace(80, 120, 100)
K, T, r, sigma = 100, 1.0, 0.05, 0.2

calls = BlackScholesEngine.calculate_prices(S, K, T, r, sigma, 'call')
puts = BlackScholesEngine.calculate_prices(S, K, T, r, sigma, 'put')

fig_bs = go.Figure()
fig_bs.add_trace(go.Scatter(x=S, y=calls, name='Call Price'))
fig_bs.add_trace(go.Scatter(x=S, y=puts, name='Put Price'))
fig_bs.update_layout(title='Black-Scholes Option Prices vs Spot', xaxis_title='Spot Price', yaxis_title='Option Value')
fig_bs.show()

# Sensitivity Analysis: Greeks
greeks = [BlackScholesEngine.calculate_greeks(s, K, T, r, sigma) for s in S]
deltas = [g['Delta'] for g in greeks]
gammas = [g['Gamma'] for g in greeks]

fig_greeks = make_subplots(rows=1, cols=2, subplot_titles=('Delta', 'Gamma'))
fig_greeks.add_trace(go.Scatter(x=S, y=deltas, name='Delta'), row=1, col=1)
fig_greeks.add_trace(go.Scatter(x=S, y=gammas, name='Gamma'), row=1, col=2)
fig_greeks.update_layout(title='Option Greeks Sensitivity')
fig_greeks.show()

## 2. Advanced Volatility: EGARCH & Regime Switching
EGARCH models capture asymmetric volatility, where market drops increase volatility more than gains, while Markov Regime Switching identifies shifts between distinct market states like bull and bear markets. These combined tools allow for a sophisticated understanding of how volatility clusters and evolves during different market phases.

In [4]:
from market_risk.volatility import GARCHEngine, RegimeSwitchingEngine

np.random.seed(42)
returns = np.random.normal(0, 0.01, 1000)
returns[500:600] *= 5 # High vol regime
returns_series = pd.Series(returns)

# EGARCH analysis
egarch = GARCHEngine()
egarch.fit(returns_series, model_type='EGARCH')
print(f'EGARCH Gamma (Leverage Effect): {egarch.gamma:.4f}')

# Regime Switching analysis
regime_eng = RegimeSwitchingEngine(k_regimes=2)
regime_eng.fit(returns_series)
probs = regime_eng.get_regime_probabilities()

fig_regime = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, subplot_titles=('Returns', 'High Vol Regime Probability'))
fig_regime.add_trace(go.Scatter(y=returns, name='Returns'), row=1, col=1)
fig_regime.add_trace(go.Scatter(y=probs.iloc[:, 1], name='Regime 1 Prob', fill='tozeroy'), row=2, col=1)
fig_regime.update_layout(height=600, title='Markov Regime Switching Analysis')
fig_regime.show()

EGARCH Gamma (Leverage Effect): 0.0000


## 3. Portfolio Risk: Copulas & Decomposition
Gaussian Copulas model the dependency between multiple assets, capturing how they move together regardless of their individual distributions. Risk Decomposition then breaks down total portfolio Value at Risk into individual asset contributions, allowing managers to identify and mitigate specific sources of risk.

In [5]:
from portfolio_risk.decomposition import RiskDecomposer, CopulaEngine

weights = np.array([0.4, 0.3, 0.3])
cov = np.array([
    [0.0004, 0.0001, 0.0002],
    [0.0001, 0.0009, 0.0003],
    [0.0002, 0.0003, 0.0006]
])

decomposer = RiskDecomposer(weights, cov)
mvar = decomposer.calculate_marginal_var(0.99)
cvar = decomposer.calculate_component_var(0.99)

fig_risk = go.Figure(data=[go.Pie(labels=['Asset A', 'Asset B', 'Asset C'], values=cvar, hole=.3)])
fig_risk.update_layout(title='Component VaR Distribution (Risk Contribution)')
fig_risk.show()

# Copula dependency visualization
corr = np.array([[1.0, 0.8], [0.8, 1.0]])
copula = CopulaEngine(corr)
samples = copula.generate_gaussian_copula_samples(2000)
fig_copula = px.scatter(x=samples[:, 0], y=samples[:, 1], opacity=0.5, title='Gaussian Copula Dependency Structure')
fig_copula.show()

## 4. Liquidity-Adjusted Risk & Stress Testing
Liquidity-Adjusted VaR (L-VaR) accounts for market friction by incorporating bid-ask spreads and price impact into risk estimates. Stress testing and correlation tilting simulate extreme market crashes and the breakdown of diversification to evaluate a portfolio's resilience under fire.

In [6]:
from market_risk.liquidity import LiquidityRiskEngine
from market_risk.stress_testing import FactorStresser

liquidity = LiquidityRiskEngine(position_size=5000, mid_price=100)
spreads = np.linspace(0.001, 0.02, 20)
l_vars = [liquidity.calculate_l_var(10000, s) for s in spreads]

fig_liq = go.Figure()
fig_liq.add_trace(go.Scatter(x=spreads, y=l_vars, name='L-VaR'))
fig_liq.update_layout(title='L-VaR Sensitivity to Bid-Ask Spread', xaxis_title='Spread', yaxis_title='Adjusted VaR')
fig_liq.show()

# Correlation stress
corr_orig = np.array([[1.0, 0.3], [0.3, 1.0]])
stressed_corr = FactorStresser.tilt_correlation(corr_orig, 2.5)
print('Original Correlation:\n', corr_orig)
print('Stressed Correlation (2.5x tilt):\n', stressed_corr)

Original Correlation:
 [[1.  0.3]
 [0.3 1. ]]
Stressed Correlation (2.5x tilt):
 [[1.   0.75]
 [0.75 1.  ]]


## 5. Credit Risk: WWR, CVA & Migration
Credit Value Adjustment (CVA) prices the risk of counterparty default, with Wrong-Way Risk (WWR) specifically modeling the dangerous correlation between exposure and credit quality. Rating Migration simulations then track the stochastic evolution of credit ratings to estimate potential losses from credit downgrades over time.

In [7]:
from credit_risk.counterparty import RatingMigrationEngine, CounterpartyRiskEngine

# Rating Migration paths
tm = np.array([[0.9, 0.08, 0.02], [0.1, 0.8, 0.1], [0.05, 0.15, 0.8]])
mig_eng = RatingMigrationEngine(tm)
n_paths = 5
fig_mig = go.Figure()
for i in range(n_paths):
    path = mig_eng.simulate_migration(0, 20)
    fig_mig.add_trace(go.Scatter(y=path, mode='lines+markers', name=f'Counterparty {i+1}'))
fig_mig.update_layout(title='Simulated Credit Rating Migration Paths', yaxis=dict(tickvals=[0, 1, 2], ticktext=['AAA', 'BBB', 'CCC']))
fig_mig.show()

# WWR Impact
t_grid = np.linspace(0, 1, 10)
cpty = CounterpartyRiskEngine(t_grid)
cpty.set_portfolio_paths(np.random.normal(0, 10, (100, 10)))
pd_curv = np.linspace(0, 0.05, 10)
alphas = np.linspace(1.0, 1.5, 10)
wwr_cvas = [cpty.calculate_cva_wwr(0.4, pd_curv, a) for a in alphas]

fig_wwr = go.Figure()
fig_wwr.add_trace(go.Scatter(x=alphas, y=wwr_cvas))
fig_wwr.update_layout(title='CVA Sensitivity to Wrong-Way Risk Alpha', xaxis_title='Alpha Multiplier', yaxis_title='Adjusted CVA')
fig_wwr.show()

## 6. Real-time Market Data Integration
This module automates the fetching of live financial data from APIs like Yahoo Finance to keep risk models updated with current market conditions. It provides a seamless data pipeline that powers the entire analytical suite with real-world price action.

In [8]:
from data.data_connectors import YahooFinanceConnector

try:
    prices = YahooFinanceConnector.fetch_historical_prices(['SPY', 'TLT'], '2023-01-01', '2024-01-01')
    fig_data = px.line(prices, title='Market Data Fetch: SPY vs TLT')
    fig_data.show()
except Exception as e:
    print(f'Data fetch skipped (possibly network/API limits): {e}')

# Part II: Real-World Portfolio Analytics Dashboard
In this section, we apply the `quant_risk_core` engine to a live, diversified portfolio of Equities, Crypto, and Commodities.

### 7. Multi-Asset Data Acquisition
We fetch the last 2 years of adjusted closing prices for a diversified basket: AAPL (Tech), MSFT (Tech), BTC-USD (Crypto), GLD (Gold), and SPY (Market). This provides a broad dataset to test our risk models across different asset classes and volatility profiles.

In [9]:
from data.data_connectors import YahooFinanceConnector
tickers = ['AAPL', 'MSFT', 'BTC-USD', 'GLD', 'SPY']
raw_data = YahooFinanceConnector.fetch_historical_prices(tickers, '2022-01-01', '2024-05-20')
raw_data.tail()

Ticker,AAPL,BTC-USD,GLD,MSFT,SPY
Date,,,,,
2024-05-15,188.090683,66267.492188,220.889999,416.533813,517.193115
2024-05-16,188.209641,65231.582031,220.029999,414.476196,516.128967
2024-05-17,188.239380,67051.875000,223.660004,413.708191,516.870972
2024-05-18,NaN,66940.804688,NaN,NaN,NaN
2024-05-19,NaN,66278.367188,NaN,NaN,NaN


### 8. Returns Engineering & Sanitization
Using the `DataPreprocessor`, we convert raw prices into continuous log-returns while handling missing values and ensuring data stationarity. Log-returns are preferred in risk modeling because they are additive over time and more closely follow a normal distribution than simple returns.

In [10]:
from market_risk.data_preprocessor import DataPreprocessor
preprocessor = DataPreprocessor()
returns_df = preprocessor.compute_log_returns(raw_data)
print(f'Returns shape: {returns_df.shape}')
returns_df.describe()

Returns shape: (867, 5)


Ticker,AAPL,BTC-USD,GLD,MSFT,SPY
count,867.000000,867.000000,867.000000,867.000000,867.000000
mean,0.000065,0.000410,0.000328,0.000287,0.000158
std,0.014697,0.029129,0.007401,0.015250,0.009651
min,-0.060472,-0.174053,-0.030161,-0.080295,-0.044456
25%,-0.005014,-0.011148,-0.002396,-0.004367,-0.002657
50%,0.000000,-0.000258,0.000000,0.000000,0.000000
75%,0.005116,0.013731,0.003075,0.005826,0.003455
max,0.085236,0.135764,0.031642,0.079059,0.053497


### 9. Comparative Performance Visualization
We visualize the cumulative growth of $1 invested in each asset to identify long-term trends and relative performance. This chart helps contextualize the risk metrics by showing the historical path and drawdown periods for each ticker.

In [11]:
cum_returns = (1 + returns_df).cumprod()
fig_cum = px.line(cum_returns, title='Cumulative Growth of $1 (2022-2024)', labels={'value': 'Portfolio Value', 'index': 'Date'})
fig_cum.show()

### 10. Multi-Asset Volatility Clustering
We fit GARCH(1,1) models to each asset to compare their conditional volatility over time. This highlights the 'Volatility Clustering' phenomenon where high-volatility periods are followed by high-volatility, particularly evident in the Crypto vs. Equity comparison.

In [12]:
from market_risk.volatility import GARCHEngine

vols_dict = {}
for ticker in tickers:
    g = GARCHEngine()
    g.fit(returns_df[ticker])
    vols_dict[ticker] = g.conditional_volatility()

vols_df = pd.DataFrame(vols_dict)
fig_vols = px.line(vols_df, title='GARCH(1,1) Conditional Volatility Comparison', labels={'value': 'Volatility', 'index': 'Date'})
fig_vols.show()

C:\Users\thoma\quant-risk-core\market_risk\volatility.py:34: ConvergenceWarning:

The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.




### 11. Market Risk Dashboard: Rolling 99% VaR
This dashboard maps the daily 99% Value at Risk for Bitcoin against its actual returns. It provides a visual validation of how well the GARCH-adjusted thresholds respond to rapid price movements and extreme market shocks.

In [13]:
from market_risk.estimators import RiskEngine
target_asset = 'BTC-USD'
risk_eng = RiskEngine(confidence_levels=[0.99])
btc_vars = [risk_eng.parametric_var_es(0, v, 'normal')['VaR_0.99'] for v in vols_df[target_asset]]

fig_btc_risk = go.Figure()
fig_btc_risk.add_trace(go.Scatter(y=returns_df[target_asset], name='BTC Returns', opacity=0.4))
fig_btc_risk.add_trace(go.Scatter(y=-np.array(btc_vars), name='99% VaR Limit', line=dict(color='red')))
fig_btc_risk.update_layout(title=f'Bitcoin Daily Returns vs 99% VaR Limit', yaxis_title='Return')
fig_btc_risk.show()

### 12. Cross-Asset Correlation Dynamics
We generate a correlation matrix to understand the diversification benefits of the portfolio. Low or negative correlations between assets like Gold (GLD) and Equities (SPY) are crucial for reducing total portfolio variance during market stress.

In [14]:
corr_matrix = returns_df.corr()
fig_corr = px.imshow(corr_matrix, text_auto=True, color_continuous_scale='RdBu_r', title='Multi-Asset Correlation Matrix')
fig_corr.show()

### 13. Portfolio Risk Contribution (Component VaR)
Assuming an equal-weighted portfolio, we decompose the total risk into individual asset contributions. This identifies which assets are 'risk-drivers' and which are 'diversifiers' based on their stand-alone volatility and their correlation with the rest of the basket.

In [15]:
from portfolio_risk.decomposition import RiskDecomposer
n_assets = len(tickers)
weights = np.array([1/n_assets] * n_assets)
cov_matrix = returns_df.cov()

decomposer = RiskDecomposer(weights, cov_matrix.values)
cvar_contributions = decomposer.calculate_component_var(0.99)

fig_port_risk = go.Figure(data=[go.Bar(x=tickers, y=cvar_contributions, marker_color='indigo')])
fig_port_risk.update_layout(title='Portfolio Component VaR: Contribution by Asset', yaxis_title='Risk Contribution')
fig_port_risk.show()

### 14. Scenario Analysis: Stressing the Portfolio
We simulate a 'Crypto Contagion' scenario where Bitcoin drops 20% while Gold rises 5% as a safe haven. This hypothetical stress test evaluates how the total portfolio value would respond to extreme, non-linear market movements.

In [16]:
from market_risk.stress_testing import ScenarioEngine
scenario_eng = ScenarioEngine(raw_data.tail(1))
shocks = {'BTC-USD': -0.20, 'GLD': 0.05}
stressed_prices = scenario_eng.apply_hypothetical_scenario(shocks)

print('Original Portfolio Prices (Recent):\n', raw_data.tail(1))
print('\nStressed Portfolio Prices (20% BTC Crash, 5% GLD Rally):\n', stressed_prices)

Original Portfolio Prices (Recent):
 Ticker      AAPL       BTC-USD  GLD  MSFT  SPY
Date                                          
2024-05-19   NaN  66278.367188  NaN   NaN  NaN

Stressed Portfolio Prices (20% BTC Crash, 5% GLD Rally):
 Ticker      AAPL      BTC-USD  GLD  MSFT  SPY
Date                                         
2024-05-19   NaN  53022.69375  NaN   NaN  NaN


### 15. Final Risk Summary Dashboard
We aggregate the key risk metrics-Annualized Volatility, 99% VaR, and Expected Shortfall-for all assets into a single dashboard. This provides a clear, actionable comparison of the risk-return profile across the entire investment universe.

In [17]:
summary_stats = []
for ticker in tickers:
    rets = returns_df[ticker]
    ann_vol = rets.std() * np.sqrt(252)
    risk = risk_eng.historical_var_es(rets)
    summary_stats.append({
        'Ticker': ticker,
        'Ann. Volatility': f'{ann_vol:.2%}',
        '99% VaR': f'{risk["VaR_0.99"]:.2%}',
        '99% ES': f'{risk["ES_0.99"]:.2%}'
    })

summary_df = pd.DataFrame(summary_stats)
fig_table = go.Figure(data=[go.Table(
    header=dict(values=list(summary_df.columns), fill_color='paleturquoise', align='left'),
    cells=dict(values=[summary_df[c] for c in summary_df.columns], fill_color='lavender', align='left'))
])
fig_table.update_layout(title='Final Risk Summary Dashboard')
fig_table.show()